# Phase 2D - Z.AI Judge Consistency (Colab)

Calibrate the Z.AI GLM judge against the existing Fireworks GLM judge. This notebook reuses saved answers and makes no retrieval or generation calls.

Both P2 and P2-1S use the same frozen 20-question judge subset. Z.AI results are stored separately; the Fireworks reference files are never modified.


In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='cf1ec06ba072dddcf25d83465aa82c2ab93d1960'
SOURCE_ZIP=Path('/content/drive/MyDrive/newsqa_phase2/phase2d_p2_one_shot_results.zip')
RUN_ID='phase2d_z_ai_judge_consistency'
EXECUTE_Z_AI_JUDGE=False  # Set True after configuring the Colab secret.
Z_AI_JUDGE_MODEL='glm-5.3-flash'
Z_AI_BASE_URL='https://api.z.ai/api/paas/v4/'
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
SEED=42
METRICS=['answer_correctness','faithfulness','answer_relevancy','context_precision','context_recall']
ARMS={'p2_d5':'one_shot_screening__p2__d5','p2_1s_d5':'one_shot_screening__p2_1s__d5'}
MEAN_DELTA_LIMIT=0.03
MAE_LIMIT=0.10
SPEARMAN_MIN=0.80
DISCRETE_AGREEMENT_MIN=0.85


## 1. Environment and immutable source

Required Colab secret: `Z_AI_API_KEY`. A GPU is optional but speeds up the local answer-relevancy embeddings.


In [ ]:
import hashlib, json, math, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
RUNTIME_ROOT=Path('/content'); PROJECT_ROOT=RUNTIME_ROOT/'Text-Mining---NewsQA-RAG'; WORK_ROOT=RUNTIME_ROOT/RUN_ID
SOURCE_ROOT=WORK_ROOT/'source'; RUNS_ROOT=WORK_ROOT/'runs'; IDS_ROOT=WORK_ROOT/'question_ids'; RESULTS=WORK_ROOT/'results'; LOGS=WORK_ROOT/'logs'
DRIVE_OUTPUT=Path('/content/drive/MyDrive/newsqa_phase2'); DRIVE_OUTPUT.mkdir(parents=True,exist_ok=True)
def secret(name):
    try: return (userdata.get(name) or '').strip()
    except Exception: return ''
assert SOURCE_ZIP.exists(),f'Upload the screening ZIP to {SOURCE_ZIP}'
for path in [SOURCE_ROOT,RUNS_ROOT,IDS_ROOT,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive: archive.extractall(SOURCE_ROOT)
assert not REPO_COMMIT.startswith('SET_TO_'),'Commit and pin this notebook before execution'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
os.environ['PYTHONPATH']=os.pathsep.join([str(PROJECT_ROOT/'common'),str(PROJECT_ROOT/'app/backend'),os.environ.get('PYTHONPATH','')]).rstrip(os.pathsep)
os.environ.update({'HF_HOME':str(RUNTIME_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false'})
Z_AI_API_KEY=secret('Z_AI_API_KEY')
if EXECUTE_Z_AI_JUDGE:
    assert Z_AI_API_KEY,'Configure the Z_AI_API_KEY Colab secret'
print('Source SHA-256:',hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest())


In [ ]:
import pandas as pd, numpy as np
from IPython.display import display
def load_jsonl(path): return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
def write_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def write_jsonl(path,rows): Path(path).write_text(''.join(json.dumps(row,sort_keys=True)+'\n' for row in rows),encoding='utf-8')
def latest_success(path):
    result={}
    for row in load_jsonl(path):
        if row.get('status')=='success': result[row['question_id']]=row
    return result
def run_command(command,label,env_overrides=None):
    command=[str(value) for value in command]; log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        env=os.environ.copy(); env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
    return log_path
def checkpoint():
    target=DRIVE_OUTPUT/f'{RUN_ID}_checkpoint.zip'; temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for root_name in ['runs','question_ids','results','logs']:
            root=WORK_ROOT/root_name
            for path in root.rglob('*'):
                if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(target); print('Checkpoint:',target,round(target.stat().st_size/2**20,1),'MiB'); return target



## 2. Validate and isolate the Fireworks reference runs


In [ ]:
judge_ids=json.loads((SOURCE_ROOT/'question_ids/judge_calibration.json').read_text()); assert len(judge_ids)==20 and len(set(judge_ids))==20
reference={}; calibration_runs={}
for arm,folder in ARMS.items():
    source_run=SOURCE_ROOT/'runs'/folder; assert source_run.exists(),source_run
    source_judges=latest_success(source_run/'judge_results.jsonl'); assert set(judge_ids)<=set(source_judges)
    assert all(set(row['scores'])==set(METRICS) for qid,row in source_judges.items() if qid in set(judge_ids))
    reference[arm]={qid:source_judges[qid] for qid in judge_ids}
    target=RUNS_ROOT/arm
    if not target.exists(): shutil.copytree(source_run,target)
    calibration_runs[arm]=target
write_json(IDS_ROOT/'judge_calibration.json',judge_ids)
display(pd.DataFrame([{'arm':arm,'questions':len(rows),'reference_provider':next(iter(rows.values()))['judge_provider'],'reference_model':next(iter(rows.values()))['judge_model']} for arm,rows in reference.items()]))


## 3. Run the Z.AI judge

Each arm is judged with the same model, metric set, output budget, retry policy, and frozen question IDs. Results are written to a separate JSONL file so the Fireworks reference remains immutable.


In [ ]:
def run_z_ai_arm(arm,run_dir):
    ids_path=IDS_ROOT/f'{arm}_z_ai.json'; write_json(ids_path,judge_ids)
    result_file='judge_results_z_ai.jsonl'; attempts_file='judge_attempts_z_ai.jsonl'
    command=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',run_dir,'--judge-provider','z_ai','--judge-model',Z_AI_JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING_EFFORT,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',ids_path,'--results-file',result_file,'--attempts-file',attempts_file,'--metrics',*METRICS,'--batch-size',1,'--max-workers',1,'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
    try:
        run_command(command,f'{arm}_z_ai',{'Z_AI_API_KEY':Z_AI_API_KEY,'Z_AI_BASE_URL':Z_AI_BASE_URL})
    except Exception:
        checkpoint(); raise
    rows=latest_success(run_dir/result_file); missing=[qid for qid in judge_ids if qid not in rows]; assert not missing,f'Incomplete {arm}/Z.AI judge: {missing[:5]}'
    write_json(run_dir/'z_ai_judge_manifest.json',{'schema_version':1,'arm':arm,'provider':'z_ai','model':Z_AI_JUDGE_MODEL,'base_url':Z_AI_BASE_URL,'reasoning_effort_requested':JUDGE_REASONING_EFFORT,'provider_thinking_mode':'enabled','questions':len(judge_ids),'metrics':METRICS,'secrets_recorded':False})
    return result_file
assert EXECUTE_Z_AI_JUDGE,'Set EXECUTE_Z_AI_JUDGE=True after validating the source and secret'
candidate={}
for arm,run_dir in calibration_runs.items(): candidate[arm]=latest_success(run_dir/run_z_ai_arm(arm,run_dir)); checkpoint()


## 4. Paired consistency analysis


In [ ]:
question_rows=[]
for arm in ARMS:
    predictions={row['question_id']:row for row in load_jsonl(calibration_runs[arm]/'predictions.jsonl')}
    assert set(judge_ids)==set(candidate[arm])
    for qid in judge_ids:
        article=predictions[qid]['article_key']
        for metric in METRICS:
            ref=float(reference[arm][qid]['scores'][metric]); cand=float(candidate[arm][qid]['scores'][metric])
            question_rows.append({'arm':arm,'question_id':qid,'article_key':article,'metric':metric,'fireworks':ref,'z_ai':cand,'delta_z_ai_minus_fireworks':cand-ref,'absolute_error':abs(cand-ref)})
question_frame=pd.DataFrame(question_rows); question_frame.to_csv(RESULTS/'judge_consistency_question_level.csv',index=False)
def cluster_bootstrap_ci(group,n_boot=5000):
    article_delta=group.groupby('article_key')['delta_z_ai_minus_fireworks'].mean().to_numpy(); rng=np.random.default_rng(SEED)
    boot=np.array([rng.choice(article_delta,size=len(article_delta),replace=True).mean() for _ in range(n_boot)])
    return np.quantile(boot,[0.025,0.975])
summary=[]
for (arm,metric),group in question_frame.groupby(['arm','metric']):
    lo,hi=cluster_bootstrap_ci(group); rho=group['fireworks'].corr(group['z_ai'],method='spearman')
    discrete=metric in {'faithfulness','context_precision','context_recall'}
    within=(group['absolute_error']<=0.10+1e-12).mean()
    gate=abs(group['delta_z_ai_minus_fireworks'].mean())<=MEAN_DELTA_LIMIT and group['absolute_error'].mean()<=MAE_LIMIT and (not math.isnan(rho) and rho>=SPEARMAN_MIN) and (not discrete or within>=DISCRETE_AGREEMENT_MIN)
    summary.append({'arm':arm,'metric':metric,'n':len(group),'articles':group['article_key'].nunique(),'fireworks_mean':group['fireworks'].mean(),'z_ai_mean':group['z_ai'].mean(),'mean_delta':group['delta_z_ai_minus_fireworks'].mean(),'mae':group['absolute_error'].mean(),'spearman':rho,'agreement_within_0.10':within,'ci95_low':lo,'ci95_high':hi,'screening_gate_pass':gate})
summary_frame=pd.DataFrame(summary); summary_frame.to_csv(RESULTS/'judge_consistency_metric_summary.csv',index=False)
disagreements=question_frame[question_frame['absolute_error']>0.10].sort_values(['absolute_error'],ascending=False); disagreements.to_csv(RESULTS/'judge_consistency_disagreements.csv',index=False)
coverage=sum(len(candidate[arm]) for arm in ARMS)/(len(ARMS)*len(judge_ids))
write_json(RESULTS/'judge_consistency_decision.json',{'schema_version':1,'coverage':coverage,'questions_per_arm':len(judge_ids),'arms':list(ARMS),'answer_instances':len(ARMS)*len(judge_ids),'all_metric_gates_pass':bool(summary_frame['screening_gate_pass'].all()),'thresholds':{'absolute_mean_delta_max':MEAN_DELTA_LIMIT,'mae_max':MAE_LIMIT,'spearman_min':SPEARMAN_MIN,'discrete_agreement_within_0.10_min':DISCRETE_AGREEMENT_MIN},'interpretation':'Operational calibration screen only; n=20 per arm does not establish statistical equivalence.'})
display(summary_frame.sort_values(['metric','arm'])); print('Coverage:',coverage,'| disagreements > 0.10:',len(disagreements)); display(disagreements.head(30))


## 5. Export


In [ ]:
manifest={'schema_version':1,'repo_commit':REPO_COMMIT,'source_zip':SOURCE_ZIP.name,'source_sha256':hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest(),'reference_provider':'fireworks','candidate_provider':'z_ai','candidate_model':Z_AI_JUDGE_MODEL,'candidate_base_url':Z_AI_BASE_URL,'reasoning_effort_requested':JUDGE_REASONING_EFFORT,'provider_thinking_mode':'enabled','question_ids_sha256':hashlib.sha256(json.dumps(judge_ids,sort_keys=True,separators=(',',':')).encode()).hexdigest(),'arms':ARMS,'metrics':METRICS,'generated_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
write_json(RESULTS/'calibration_manifest.json',manifest)
bundle=DRIVE_OUTPUT/f'{RUN_ID}_results.zip'; temporary=bundle.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for root_name in ['runs','question_ids','results','logs']:
        root=WORK_ROOT/root_name
        for path in root.rglob('*'):
            if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
temporary.replace(bundle); checkpoint(); print('Results:',bundle,round(bundle.stat().st_size/2**20,1),'MiB')
